> ### ⚠️ These results contain look-ahead bias — do not quote them
>
> The Sharpe ratio reported below (**8.59** net of costs) is an artifact of
> this notebook's construction, not a research result. The strategy trades the
> cumulative sum of the Kalman filter's own one-step innovations, and the
> filter's update step manufactures the negative autocorrelation it then
> trades: sweeping the process-noise parameter moves Sharpe from 1.93 to 9.03.
> The OU calibration and the universe selection are also fitted on the full
> sample.
>
> This notebook is kept as the record of the original construction.
> **For the corrected, look-ahead-free pipeline see
> `05_causal_backtest_validation.ipynb`**, which reports Sharpe **0.52**
> (sign-shuffle null p = 0.110, i.e. not statistically significant).
> The full analysis is in `results/BACKTEST_FINDINGS.md`.

# Portfolio Backtest

Walk-forward statistical arbitrage backtest: rolling PCA factor extraction, UMAP/DBSCAN clustering, Kalman-filtered idiosyncratic residuals, stationarity screening, OU calibration, and portfolio construction with transaction costs.

## 1: Environment Setup & Module Imports

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

from src.backtest import PortfolioBacktester
from src.clustering import FactorClusterer
from src.data_loader import fetch_equity_returns
from src.diagnostics import SpreadDiagnostics
from src.ou_process import OUProcessModel
from src.residuals_KF import extract_idiosyncratic_residuals
from src.rolling_engine import RollingPCAEngine

# Configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (12, 6)

## 2: Data Ingestion & End-to-End Pipeline Execution

In [ ]:
# Broad, multi-sector liquid asset universe
tickers = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "META", "TSLA", "AMD", "INTC", "QCOM",
    "JPM", "BAC", "WFC", "C", "GS", "MS", "XOM", "CVX", "COP", "SLB"
]
# Extended so there is enough history *before* 2023 to actually roll a
# 2-year (504-day) estimation window forward across the 2023-2025
# backtest period, rather than fitting PCA once on the whole sample.
start_date = "2019-01-01"
end_date = "2025-01-01"
backtest_start = "2023-01-01"  # the period we actually trade/evaluate

# 1. Fetch Returns Matrix
returns = fetch_equity_returns(tickers, start_date=start_date, end_date=end_date)
print(f"Returns Matrix Shape: {returns.shape}")

# 2. Walk-Forward PCA: re-fit the top 5 systematic factors every 21
#    trading days (~monthly) on a trailing 504-day (~2yr) window, and
#    project each day's return onto the eigenvectors estimated as of
#    the PRIOR rebalance -- no look-ahead into the loadings.
rolling_pca = RollingPCAEngine(n_components=5, window=504, rebalance_freq=21)
factor_returns = rolling_pca.fit_transform(returns)
print(f"Rolling factor returns shape: {factor_returns.shape}")
print(f"Number of PCA re-estimations: {len(rolling_pca.rebalance_dates_)}")

# Restrict to the actual backtest period now that the factor space has
# been walk-forward estimated using the pre-2023 history as burn-in.
returns = returns.loc[backtest_start:]
factor_returns = factor_returns.loc[backtest_start:]

# 3. Dynamic Asset Clustering (Parametric UMAP + DBSCAN)
#    Re-fitting the neural UMAP encoder on every 21-day PCA rebalance
#    would be expensive, so clustering uses the most recent factor
#    loadings snapshot as of the start of the backtest window.
clusterer = FactorClusterer(n_components=2, eps=0.4, min_samples=2)
clusterer.fit(rolling_pca.loadings_as_of(returns.index[0]))
clusters = clusterer.get_clusters()

# 4. Dynamic Idiosyncratic Residual Extraction via Kalman Filter
#    (the KF was already adaptive day-to-day; it now also tracks a
#    factor space that itself walks forward instead of staying fixed)
residuals, cumulative_spreads, _ = extract_idiosyncratic_residuals(
    returns, factor_returns, process_noise=1e-4, measurement_noise=1e-3
)

# Truncate 30-day Kalman Filter burn-in period
burn_in = 30
clean_residuals = residuals.iloc[burn_in:]
clean_spreads = cumulative_spreads.iloc[burn_in:]

## 3: Diagnostics Gatekeeper & Continuous OU Calibration

In [ ]:
diag = SpreadDiagnostics(significance_level=0.05)
diagnostic_summary = diag.filter_tradeable_spreads(clean_spreads, clean_residuals)

# Identify tradeable assets that are NOT classified as noise (-1) in DBSCAN
noise_assets = set(clusters.get(-1, []))
tradeable_mask = diagnostic_summary["tradeable"] & (~diagnostic_summary.index.isin(noise_assets))
active_universe = diagnostic_summary[tradeable_mask].index.tolist()

print(f"Active Tradeable Universe: {len(active_universe)} / {len(tickers)} assets")
print(active_universe)

# Fit Ornstein-Uhlenbeck Process & Compute S-Scores
ou_engine = OUProcessModel()
s_scores = pd.DataFrame(index=clean_spreads.index, columns=active_universe)
sigma_eq_dict = {}

for ticker in active_universe:
    params = ou_engine.fit_spread(clean_spreads[ticker])
    if not np.isnan(params["sigma_eq"]):
        sigma_eq_dict[ticker] = params["sigma_eq"]
        s_scores[ticker] = ou_engine.compute_s_score(clean_spreads[ticker])

s_scores = s_scores.dropna(how="all", axis=1).astype(float)

## 4: Multi-Asset Execution & Transaction Cost Backtest

In [ ]:
# Initialize backtester with 5 bps slippage/commissions per trade
backtester = PortfolioBacktester(
    s_open=1.25,
    s_close=0.5,
    transaction_cost_bps=5.0,
    max_gross_leverage=1.0
)

# 1. Signal Generation
signals = backtester.generate_signals(s_scores)

# 2. Inverse-Volatility Risk Parity Allocation
weights = backtester.compute_portfolio_weights(signals, sigma_eq_dict)

# 3. Simulate Execution
results = backtester.run_backtest(weights, clean_residuals[s_scores.columns])

# Fetch SPY benchmark for relative comparison
spy_prices = yf.download("SPY", start=clean_spreads.index[0], end=clean_spreads.index[-1])["Close"]
spy_returns = spy_prices.pct_change().dropna()
spy_equity = (1.0 + spy_returns).cumprod()

## 5: Performance Tearsheet & Metrics Breakdown

In [ ]:
net_metrics = backtester.calculate_metrics(results["net_returns"])
gross_metrics = backtester.calculate_metrics(results["gross_returns"])

metrics_df = pd.DataFrame({
    "Gross Strategy": gross_metrics,
    "Net Strategy (5 bps TC)": net_metrics
})

print("=== Institutional Performance Tearsheet ===")
print(metrics_df.round(4))

## 6: Visualizations — Equity Curve, Underwater Drawdown & Exposure

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(13, 11), sharex=True, gridspec_kw={"height_ratios": [3, 1.5, 1.5]})

# 1. Cumulative Equity Curves
ax1.plot(results.index, results["gross_equity"], label="Gross Strategy (No TC)", color="gray", ls="--", alpha=0.8)
ax1.plot(results.index, results["net_equity"], label="Net Strategy (5 bps TC)", color="navy", lw=2)
# Align SPY to start at 1.0 on the same date
spy_aligned = spy_equity.reindex(results.index, method="ffill")
spy_normalized = spy_aligned / spy_aligned.iloc[0]
ax1.plot(results.index, spy_normalized, label="S&P 500 Benchmark (SPY)", color="crimson", lw=1.5, alpha=0.7)
ax1.set_ylabel("Portfolio Value (Base 1.0)")
ax1.set_title("Machine Learning-Enhanced Statistical Arbitrage: Portfolio Backtest")
ax1.legend(loc="upper left")

# 2. Underwater Drawdown Curve
cum_net = results["net_equity"]
running_max = cum_net.cummax()
drawdown = (cum_net - running_max) / running_max
ax2.fill_between(drawdown.index, drawdown, 0, color="crimson", alpha=0.3)
ax2.plot(drawdown.index, drawdown, color="crimson", lw=1)
ax2.set_ylabel("Drawdown")
ax2.set_title("Strategy Underwater Drawdown Curve")

# 3. Gross Leverage & Active Positions Over Time
gross_exposure = weights.abs().sum(axis=1)
ax3.plot(gross_exposure.index, gross_exposure, color="darkgreen", lw=1.5, label="Gross Leverage")
ax3.set_ylabel("Leverage (x)")
ax3.set_xlabel("Date")
ax3.set_ylim(0, 1.2)
ax3.legend(loc="upper left")

plt.tight_layout()
plt.show()